# Week 3 Guided Lab: First Steps in Data Cleaning

**BAN 6003: Data Management and Analytics Integration**

In Week 2, you acted as a data detective. You loaded a dataset, inspected its structure, checked missing values, reviewed summary statistics, and looked for duplicates.

This week, you start acting as a careful data surgeon. Your job is not to randomly change the dataset. Your job is to make deliberate, documented cleaning decisions.

In this guided lab, we will use `employee_data.csv` to practice a first cleaning pass.

## Lab Learning Goals

By the end of this lab, you should be able to:

1. Make a safe working copy of a raw DataFrame.
2. Correct common data type problems using `pd.to_numeric()` and `pd.to_datetime()`.
3. Understand what `errors="coerce"` does.
4. Identify full-row duplicates and key-based duplicates.
5. Remove duplicate employee IDs when `employee_id` should be unique.
6. Compare missing-data strategies: dropping and imputation.
7. Impute missing numeric values using the median.
8. Drop rows with missing critical identifiers or dates.
9. Verify the cleaned dataset using `info()`, `shape`, and `isna().sum()`.
10. Keep a simple cleaning log that documents your decisions.

## Important: Cleaning Is Not Guessing

In practice, we need to be careful. A cleaning decision should be based on the business meaning of the column.

For example, if `employee_id` is the unique identifier for each employee, then a duplicate employee ID is a serious issue. If `salary` is missing, we may be able to impute it in a classroom exercise, but in a real HR or financial services environment we might need to investigate the source system before filling it in.

This lab teaches the mechanics. Your project will require you to explain the reasoning.

## Part 1: Import Pandas and Load the Raw Data

We will start by importing Pandas and loading the same type of employee dataset used in the Week 2 profiling activity.

In [ ]:
import pandas as pd

# Load the raw employee data
raw_path = "../data/employee_data.csv"
df = pd.read_csv(raw_path)

df.head()

## Part 2: Review the Raw Data

Before cleaning anything, inspect the dataset again. We want to confirm the shape, data types, missing values, and suspicious values.

In [ ]:
# How many rows and columns?
df.shape

In [ ]:
# What data types did Pandas assign?
df.info()

In [ ]:
# Count missing values by column
df.isna().sum()

In [ ]:
# First look at all rows for this small demo dataset
df

### Think Before Cleaning

Look at the output above. You should notice several issues:

- `employee_id` should be numeric, but one row contains the text value `error`.
- `hire_date` should be a date, but at least one value is not a valid date.
- `salary` contains dollar signs, commas, blanks, and invalid text.
- `age` contains missing or invalid values.
- `employee_id` 102 appears more than once.
- Some values may be valid technically but suspicious, such as an age of 150.

Today we will focus on structural errors, duplicates, and missing values. We will not fully standardize categories yet.

## Part 3: Create a Safe Working Copy

Never overwrite your raw data immediately. Keep the original DataFrame unchanged and create a working copy.

This is one of the most important habits in data cleaning.

In [ ]:
# Keep the raw df unchanged and clean a copy
raw_df = df.copy()
df_clean = df.copy()

# Start a simple cleaning log
cleaning_log = []

print("Raw data shape:", raw_df.shape)
print("Working copy shape:", df_clean.shape)

## Part 4: Correct Numeric Data Types

The first structural problem is `employee_id`. It should be numeric, but one row contains the text value `error`.

We will use `pd.to_numeric()` with `errors="coerce"`.

When we use `errors="coerce"`, any value that cannot be converted becomes a missing value, shown as `NaN` or `<NA>`. This is useful because it exposes invalid values instead of silently hiding them.

In [ ]:
# Convert employee_id to a numeric value.
# Invalid values such as "error" become NaN.
df_clean["employee_id"] = pd.to_numeric(df_clean["employee_id"], errors="coerce")

# Use pandas nullable integer type so missing IDs can exist temporarily
df_clean["employee_id"] = df_clean["employee_id"].astype("Int64")

cleaning_log.append("Converted employee_id to numeric using pd.to_numeric(errors='coerce'). Invalid IDs became missing values.")

df_clean[["employee_id"]]

### Cleaning Salary Values

The `salary` column is trickier because it may contain values like `$95,000`. If we directly convert those strings to numbers, Pandas may treat them as invalid because of the dollar sign and comma.

So we will first remove `$` and `,`, then convert the result to numeric.

In [ ]:
# Show the original salary values before conversion
df_clean["salary"]

In [ ]:
# Remove dollar signs and commas, then convert to numeric
salary_cleaned = (
    df_clean["salary"]
    .astype("string")
    .str.replace("$", "", regex=False)
    .str.replace(",", "", regex=False)
    .str.strip()
)

df_clean["salary"] = pd.to_numeric(salary_cleaned, errors="coerce")

cleaning_log.append("Removed dollar signs and commas from salary, then converted salary to numeric. Invalid salary values became missing values.")

df_clean[["salary"]]

### Cleaning Age Values

The `age` column should also be numeric. If values such as `thirty` appear, they should be converted to missing values during this first structural cleaning pass.

In [ ]:
df_clean["age"] = pd.to_numeric(df_clean["age"], errors="coerce")

cleaning_log.append("Converted age to numeric using pd.to_numeric(errors='coerce'). Invalid age values became missing values.")

df_clean[["age"]]

## Part 5: Correct Date Data Types

The `hire_date` column should be a date. We will use `pd.to_datetime()` with `errors="coerce"`.

Invalid dates become `NaT`, which means Not a Time. You can think of `NaT` as the date/time version of a missing value.

In [ ]:
df_clean["hire_date"] = pd.to_datetime(df_clean["hire_date"], errors="coerce")

cleaning_log.append("Converted hire_date to datetime using pd.to_datetime(errors='coerce'). Invalid dates became NaT.")

df_clean[["hire_date"]]

## Part 6: Check What Changed

After type conversion, inspect the data again. Notice how invalid values have become missing values. This is a good thing because now they can be counted and handled consistently.

In [ ]:
df_clean.info()

In [ ]:
df_clean.isna().sum()

## Part 7: Identify Duplicates

There are two common ways to think about duplicates.

A full-row duplicate means every value in the row is repeated. A key-based duplicate means a specific identifier, such as `employee_id`, appears more than once.

For this employee dataset, `employee_id` should identify one employee. So we care about key-based duplicates.

In [ ]:
# Full-row duplicate count
df_clean.duplicated().sum()

In [ ]:
# Key-based duplicate count for employee_id
df_clean["employee_id"].duplicated().sum()

In [ ]:
# Show rows with duplicated employee_id values
# keep=False marks all occurrences of the duplicate ID, not just the later ones.
df_clean[df_clean["employee_id"].duplicated(keep=False)].sort_values("employee_id")

## Part 8: Remove Key-Based Duplicates

Because `employee_id` is supposed to be unique, we will keep the first occurrence and remove later duplicate rows.

In practice, we would not always do this automatically. For example, if the two rows had conflicting salaries or hire dates, we would need to investigate. In this small lab dataset, the repeated employee 102 appears to be a duplicate record.

In [ ]:
before_rows = df_clean.shape[0]

df_clean = df_clean.drop_duplicates(subset=["employee_id"], keep="first")

after_rows = df_clean.shape[0]
rows_removed = before_rows - after_rows

cleaning_log.append(f"Removed {rows_removed} duplicate row(s) based on employee_id, keeping the first occurrence.")

print("Rows before:", before_rows)
print("Rows after:", after_rows)
print("Rows removed:", rows_removed)

## Part 9: Inspect Missing Data After Type Conversion

Now we need to decide what to do with missing values. Some missing values existed in the raw data. Others were created by `errors="coerce"` when invalid values were converted to missing values.

This is normal. Type conversion often reveals hidden data quality problems.

In [ ]:
df_clean.isna().sum()

In [ ]:
# Show rows with any missing values
missing_rows = df_clean[df_clean.isna().any(axis=1)]
missing_rows

## Part 10: Handle Missing Critical Fields by Dropping Rows

Some fields are critical. In this dataset, `employee_id` is the identifier, and `hire_date` is important for many HR analyses.

For this lab, we will drop rows with missing `employee_id` or missing `hire_date`.

This is a classroom decision. In practice, we might first go back to the HR system or source database to investigate.

In [ ]:
before_rows = df_clean.shape[0]

df_clean = df_clean.dropna(subset=["employee_id", "hire_date"])

after_rows = df_clean.shape[0]
rows_removed = before_rows - after_rows

cleaning_log.append(f"Dropped {rows_removed} row(s) with missing critical employee_id or hire_date.")

print("Rows before:", before_rows)
print("Rows after:", after_rows)
print("Rows removed:", rows_removed)

## Part 11: Impute Missing Numeric Values Using the Median

For this lab, we will impute missing `salary` and `age` values using the median.

The median is often safer than the mean when a column has outliers. For example, a single extremely high salary can pull the mean upward, but the median is more resistant to extreme values.

This does not mean median imputation is always best. It is just a simple first strategy.

In [ ]:
# Calculate medians before filling missing values
salary_median = df_clean["salary"].median()
age_median = df_clean["age"].median()

print("Salary median:", salary_median)
print("Age median:", age_median)

In [ ]:
# Fill missing salary and age values with their medians
df_clean["salary"] = df_clean["salary"].fillna(salary_median)
df_clean["age"] = df_clean["age"].fillna(age_median)

cleaning_log.append(f"Imputed missing salary values with the median salary: {salary_median}.")
cleaning_log.append(f"Imputed missing age values with the median age: {age_median}.")

df_clean[["salary", "age"]].head()

## Part 12: Check for Suspicious Values

Cleaning missing values does not mean the data is perfect. We also need to look for suspicious values.

For example, the age value 150 is technically numeric, but it is probably not realistic for an employee. We will flag it here, but we will not automatically fix it in this lab.

In [ ]:
# Show unusually high ages
suspicious_age = df_clean[df_clean["age"] > 100]
suspicious_age

In [ ]:
if not suspicious_age.empty:
    cleaning_log.append("Flagged age values above 100 for further review. These were not automatically changed in this lab.")

## Part 13: Verify the Cleaned Dataset

Now inspect the cleaned DataFrame. We want to confirm that the major structural issues were addressed.

In [ ]:
df_clean.info()

In [ ]:
df_clean.isna().sum()

In [ ]:
df_clean.shape

In [ ]:
df_clean.head()

## Part 14: Review the Cleaning Log

A cleaning log helps you remember what changed and why. It is also useful for project reports, reproducibility, and professional communication.

In [ ]:
for i, item in enumerate(cleaning_log, start=1):
    print(f"{i}. {item}")

## Part 15: Save the Cleaned Data

For this lab, we will save the cleaned dataset as a new CSV file. We are not overwriting the raw file.

In [ ]:
output_path = "../data/employee_data_cleaned_week3.csv"
df_clean.to_csv(output_path, index=False)
print(f"Cleaned data saved to: {output_path}")

## Part 16: Your Turn

Answer the following questions in the markdown cell below.

1. Which cleaning step created new missing values? Why did that happen?
2. Why did we use median imputation for `salary` and `age` instead of mean imputation?
3. Why is it safer to use `drop_duplicates(subset=["employee_id"])` instead of only `drop_duplicates()` in this dataset?
4. What issue did we flag but not automatically fix?
5. What is one cleaning decision in this lab that might require more investigation in a real organization?

### Your Answers

Write your answers here.

1. 
2. 
3. 
4. 
5.

## Submission Reminder

Before submitting:

1. Run all cells from top to bottom.
2. Make sure the notebook has no errors.
3. Save the notebook.
4. Commit and push your work to GitHub.

Suggested Git commands:

```bash
git add .
git commit -m "Completed Week 3 data cleaning lab"
git push
```